# Exploratory Analysis: Dataset Statistics (Memory-Efficient)

This notebook analyzes each CDCR dataset to understand:
1. Number of mentions per dataset
2. Dataset statistics (topics, mentions per topic, clusters, etc.)
3. Estimated pairs based on cluster structure

**Note**: This version does NOT create actual training pairs to avoid GPU memory issues.
It provides statistics and estimates instead.

## Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import time
import gc

# Add CL_COREF to path
sys.path.insert(0, os.path.join(os.getcwd(), 'CL_COREF'))

from dataset import CDCR_Dataset

print("Imports successful!")

## Configuration

In [ ]:
# Dataset configuration
BASE_DATA_PATH = "CL_COREF/data"
CONTEXT_METHOD = "hsu"
SPECIAL_MENTION_TOKENS = ("[E]", "[/E]")

# Datasets to analyze
DATASETS = [
    "CD2CR",
    "CEREC",
    "ECBplus",
    "ECBplusMETAm",
    "FCC",
    "GVC",
    "HyperCoref",
    "NP4E",
    "NewsWCL50",
    "NiDENT",
    "WECEng"
]

print(f"Will analyze {len(DATASETS)} datasets")
print(f"Datasets: {', '.join(DATASETS)}")

## Helper Functions

In [ ]:
def estimate_pairs_from_clusters(gold_clusters_df):
    """
    Estimate number of positive and negative pairs based on cluster structure.
    
    Positive pairs: n*(n-1)/2 for each cluster of size n
    Negative pairs: estimated based on sampling strategy
    """
    if gold_clusters_df.empty:
        return 0, 0, 0
    
    # Count cluster sizes
    cluster_sizes = gold_clusters_df.groupby('gold_cluster').size()
    
    # Positive pairs: sum of n*(n-1)/2 for each cluster
    positive_pairs = sum(n * (n - 1) // 2 for n in cluster_sizes if n > 1)
    
    # Negative pairs: estimate based on typical sampling ratios
    # The sampling algorithm typically creates 1-3 negative pairs per positive pair
    # This is a rough estimate
    negative_pairs = positive_pairs * 2  # Conservative estimate
    
    total_pairs = positive_pairs + negative_pairs
    
    return total_pairs, positive_pairs, negative_pairs

def analyze_cdcr_dataset(cdcr_dataset):
    """
    Analyze a CDCR_Dataset instance and return statistics.
    """
    stats = {
        'dataset_name': cdcr_dataset.name,
        'total_mentions': len(cdcr_dataset.mentions),
        'total_topics': len(cdcr_dataset.mentions_by_topic),
        'entity_mentions': len(cdcr_dataset.mentions_by_type.get('entity', [])),
        'event_mentions': len(cdcr_dataset.mentions_by_type.get('event', [])),
        'total_documents': len(cdcr_dataset.docs),
        'topics': sorted(list(cdcr_dataset.mentions_by_topic.keys())),
    }
    
    # Mentions per topic
    mentions_per_topic = [len(mentions) for mentions in cdcr_dataset.mentions_by_topic.values()]
    stats['avg_mentions_per_topic'] = np.mean(mentions_per_topic) if mentions_per_topic else 0
    stats['min_mentions_per_topic'] = min(mentions_per_topic) if mentions_per_topic else 0
    stats['max_mentions_per_topic'] = max(mentions_per_topic) if mentions_per_topic else 0
    
    # Cluster statistics from gold_clusters_df
    if not cdcr_dataset.gold_clusters_df.empty:
        stats['total_clusters'] = cdcr_dataset.gold_clusters_df['gold_cluster'].nunique()
        cluster_sizes = cdcr_dataset.gold_clusters_df.groupby('gold_cluster').size()
        stats['avg_cluster_size'] = cluster_sizes.mean()
        stats['max_cluster_size'] = cluster_sizes.max()
        stats['singletons'] = (cluster_sizes == 1).sum()
        stats['non_singleton_clusters'] = (cluster_sizes > 1).sum()
        
        # Estimate pairs
        total_pairs, positive_pairs, negative_pairs = estimate_pairs_from_clusters(cdcr_dataset.gold_clusters_df)
        stats['estimated_total_pairs'] = total_pairs
        stats['estimated_positive_pairs'] = positive_pairs
        stats['estimated_negative_pairs'] = negative_pairs
    else:
        stats['total_clusters'] = 0
        stats['avg_cluster_size'] = 0
        stats['max_cluster_size'] = 0
        stats['singletons'] = 0
        stats['non_singleton_clusters'] = 0
        stats['estimated_total_pairs'] = 0
        stats['estimated_positive_pairs'] = 0
        stats['estimated_negative_pairs'] = 0
    
    return stats

def print_dataset_stats(stats):
    """
    Pretty print dataset statistics.
    """
    print(f"\n{'='*70}")
    print(f"Dataset: {stats['dataset_name']}")
    print(f"{'='*70}")
    print(f"\nMentions: {stats['total_mentions']:,}")
    print(f"  - Entity mentions: {stats['entity_mentions']:,}")
    print(f"  - Event mentions: {stats['event_mentions']:,}")
    print(f"\nTopics: {stats['total_topics']}")
    print(f"  - Topic IDs: {', '.join(map(str, stats['topics'][:10]))}{'...' if len(stats['topics']) > 10 else ''}")
    print(f"  - Avg mentions/topic: {stats['avg_mentions_per_topic']:.1f}")
    print(f"  - Min mentions/topic: {stats['min_mentions_per_topic']}")
    print(f"  - Max mentions/topic: {stats['max_mentions_per_topic']}")
    print(f"\nClusters: {stats['total_clusters']:,}")
    print(f"  - Non-singleton clusters: {stats['non_singleton_clusters']:,}")
    print(f"  - Singletons: {stats['singletons']:,}")
    print(f"  - Avg cluster size: {stats['avg_cluster_size']:.2f}")
    print(f"  - Max cluster size: {stats['max_cluster_size']}")
    print(f"\nEstimated Training Pairs: {stats['estimated_total_pairs']:,}")
    print(f"  - Positive pairs: {stats['estimated_positive_pairs']:,}")
    print(f"  - Negative pairs: {stats['estimated_negative_pairs']:,} (estimate)")
    if stats['total_mentions'] > 0:
        print(f"  - Pairs per mention: {stats['estimated_total_pairs']/stats['total_mentions']:.2f}")
    print(f"\nDocuments: {stats['total_documents']:,}")

print("Helper functions defined!")

## Analyze Each Dataset

For each dataset, we'll:
1. Load the train split as a CDCR_Dataset
2. Analyze basic statistics
3. Estimate training pairs based on cluster structure
4. Clean up memory before next dataset

In [ ]:
results = []

for dataset_name in DATASETS:
    print(f"\n{'#'*70}")
    print(f"# Processing: {dataset_name}")
    print(f"{'#'*70}")
    
    try:
        # Path to train split
        train_path = os.path.join(BASE_DATA_PATH, dataset_name, "train")
        
        if not os.path.exists(train_path):
            print(f"⚠ Warning: Train path does not exist: {train_path}")
            continue
        
        # Load CDCR_Dataset
        print(f"\nLoading CDCR_Dataset...")
        start_time = time.time()
        cdcr_dataset = CDCR_Dataset(
            name=dataset_name,
            data_path=train_path,
            context_method=CONTEXT_METHOD,
            special_mention_tokens=SPECIAL_MENTION_TOKENS,
            predicted_topics=False,
            topic_filter=None,  # No filtering
            max_mentions=None   # No limit
        )
        load_time = time.time() - start_time
        print(f"✓ Loaded in {load_time:.2f}s")
        
        # Skip if no mentions loaded
        if len(cdcr_dataset.mentions) == 0:
            print(f"⚠ Skipping {dataset_name}: No mentions found in train split")
            # Clean up
            del cdcr_dataset
            gc.collect()
            continue
        
        # Analyze dataset
        print(f"\nAnalyzing dataset statistics...")
        stats = analyze_cdcr_dataset(cdcr_dataset)
        stats['load_time'] = load_time
        print_dataset_stats(stats)
        
        # Store results
        results.append(stats)
        
        print(f"\n✓ {dataset_name} analysis complete!")
        
        # Clean up memory before next dataset
        del cdcr_dataset
        gc.collect()
        
    except Exception as e:
        print(f"\n✗ Error processing {dataset_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        # Clean up on error
        gc.collect()
        continue

print(f"\n\n{'='*70}")
print(f"Analysis complete! Processed {len(results)}/{len(DATASETS)} datasets.")
print(f"{'='*70}")

## Summary Results Table

In [ ]:
# Create summary DataFrame
if len(results) == 0:
    print("⚠ No datasets were successfully processed. Check the errors above.")
else:
    summary_df = pd.DataFrame(results)

    # Calculate pairs per mention
    summary_df['pairs_per_mention'] = summary_df['estimated_total_pairs'] / summary_df['total_mentions']

    # Select key columns for display
    display_columns = [
        'dataset_name',
        'total_mentions',
        'total_topics',
        'total_clusters',
        'non_singleton_clusters',
        'estimated_total_pairs',
        'estimated_positive_pairs',
        'estimated_negative_pairs',
        'pairs_per_mention',
        'load_time'
    ]

    summary_display = summary_df[display_columns].copy()

    # Format for better display
    summary_display['load_time'] = summary_display['load_time'].apply(lambda x: f"{x:.2f}s")
    summary_display['pairs_per_mention'] = summary_display['pairs_per_mention'].apply(lambda x: f"{x:.2f}")

    print("\n" + "="*120)
    print("SUMMARY: Dataset Statistics and Estimated Pair Counts")
    print("="*120)
    print(summary_display.to_string(index=False))

    # Save to CSV
    output_path = "dataset_analysis_results.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")

## Visualizations

In [ ]:
if len(results) > 0:
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_style("whitegrid")

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: Total mentions per dataset
    ax1 = axes[0, 0]
    summary_df.plot(x='dataset_name', y='total_mentions', kind='bar', ax=ax1, color='steelblue', legend=False)
    ax1.set_title('Total Mentions per Dataset', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Dataset', fontsize=12)
    ax1.set_ylabel('Number of Mentions', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)

    # Plot 2: Estimated training pairs per dataset
    ax2 = axes[0, 1]
    summary_df.plot(x='dataset_name', y='estimated_total_pairs', kind='bar', ax=ax2, color='coral', legend=False)
    ax2.set_title('Estimated Training Pairs per Dataset', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Dataset', fontsize=12)
    ax2.set_ylabel('Number of Pairs', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Positive vs Negative pairs
    ax3 = axes[1, 0]
    x = np.arange(len(summary_df))
    width = 0.35
    ax3.bar(x - width/2, summary_df['estimated_positive_pairs'], width, label='Positive', color='green', alpha=0.7)
    ax3.bar(x + width/2, summary_df['estimated_negative_pairs'], width, label='Negative (est.)', color='red', alpha=0.7)
    ax3.set_title('Estimated Positive vs Negative Pairs per Dataset', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Dataset', fontsize=12)
    ax3.set_ylabel('Number of Pairs', fontsize=12)
    ax3.set_xticks(x)
    ax3.set_xticklabels(summary_df['dataset_name'], rotation=45, ha='right')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)

    # Plot 4: Pairs per mention ratio
    ax4 = axes[1, 1]
    summary_df.plot(x='dataset_name', y='pairs_per_mention', kind='bar', ax=ax4, color='purple', legend=False)
    ax4.set_title('Estimated Pairs per Mention Ratio', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Dataset', fontsize=12)
    ax4.set_ylabel('Pairs per Mention', fontsize=12)
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('dataset_analysis_plots.png', dpi=300, bbox_inches='tight')
    print("✓ Plots saved to: dataset_analysis_plots.png")
    plt.show()
else:
    print("⚠ No results to visualize")

## Key Findings

In [ ]:
if len(results) > 0:
    print("\n" + "="*80)
    print("KEY FINDINGS")
    print("="*80)

    # Total statistics
    total_mentions = summary_df['total_mentions'].sum()
    total_pairs = summary_df['estimated_total_pairs'].sum()
    avg_pairs_per_mention = summary_df['pairs_per_mention'].mean()

    print(f"\n1. OVERALL STATISTICS:")
    print(f"   - Total datasets analyzed: {len(summary_df)}")
    print(f"   - Total mentions across all datasets: {total_mentions:,}")
    print(f"   - Estimated total training pairs: {total_pairs:,}")
    print(f"   - Average pairs per mention: {avg_pairs_per_mention:.2f}")

    # Largest datasets
    largest_by_mentions = summary_df.nlargest(3, 'total_mentions')
    print(f"\n2. LARGEST DATASETS (by mentions):")
    for idx, row in largest_by_mentions.iterrows():
        print(f"   - {row['dataset_name']}: {row['total_mentions']:,} mentions, ~{row['estimated_total_pairs']:,} pairs")

    # Most pairs generated
    most_pairs = summary_df.nlargest(3, 'estimated_total_pairs')
    print(f"\n3. MOST TRAINING PAIRS ESTIMATED:")
    for idx, row in most_pairs.iterrows():
        print(f"   - {row['dataset_name']}: ~{row['estimated_total_pairs']:,} pairs ({row['pairs_per_mention']:.2f} per mention)")

    # Efficiency (pairs per mention)
    most_efficient = summary_df.nlargest(3, 'pairs_per_mention')
    print(f"\n4. HIGHEST PAIRS-PER-MENTION RATIO:")
    for idx, row in most_efficient.iterrows():
        print(f"   - {row['dataset_name']}: {row['pairs_per_mention']:.2f} pairs/mention")

    print(f"\n5. RECOMMENDATIONS:")
    print(f"   - For balanced multi-dataset training, use max_mentions to equalize dataset sizes")
    print(f"   - Datasets with high pairs-per-mention ratio may dominate training")
    print(f"   - Consider topic filtering for very large datasets to reduce training time")
    print(f"   - Note: Pair counts are ESTIMATES based on cluster structure")
    print(f"   - Actual pair counts may vary based on sampling algorithm parameters")

    print(f"\n" + "="*80)
else:
    print("⚠ No results to analyze")

## Suggested Configurations

In [ ]:
if len(results) > 0:
    print("\n" + "="*80)
    print("SUGGESTED CONFIGURATIONS FOR data_setups.yaml")
    print("="*80)

    # Suggest balanced configurations
    median_mentions = summary_df['total_mentions'].median()

    print(f"\n# Balanced configuration (equalizing to median: {int(median_mentions)} mentions)\n")
    setup_id = 10
    for idx, row in summary_df.iterrows():
        if row['total_mentions'] > median_mentions * 1.5:
            suggested_max = int(median_mentions)
            print(f"{setup_id}:")
            print(f"  name: \"{row['dataset_name'].lower()}_balanced\"")
            print(f"  datasets:")
            print(f"    - name: \"{row['dataset_name']}\"")
            print(f"      splits: [\"train\", \"val\"]")
            print(f"      topic_filter: null")
            print(f"      max_mentions: {suggested_max}  # Original: {row['total_mentions']:,}")
            print()
            setup_id += 1

    print("\n✓ Analysis complete!")
else:
    print("⚠ No results to generate configurations")